# Multi-agent
When one agent's context grows unwieldy, split the work across specialists — routed through a supervisor, or handed off by updating state.

## Concept
- Subagents — a supervisor calls specialists as tools; all routing passes through it.
- Handoffs — a tool call updates state to transfer control to another agent.
- Router — a classifier sends input to the right specialist, then synthesizes.
- Skills — one agent stays in control, loading extra context on demand.

In [ ]:
from pydantic import SecretStr
import os
import math

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool

@tool
def web_search(query: str) -> str:
    """Search the web for real-time information, market data, or general knowledge."""
    # Dummy implementation
    return f"Mock search results for '{query}': Revenue figures for last year show strong 14% YOY growth across target sectors."

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression safely (e.g., '1500000 * 0.12')."""
    try:
        # Using a restricted evaluation for basic arithmetic
        allowed_globals = {"__builtins__": None}
        allowed_locals = {"math": math}
        result = eval(expression, allowed_globals, allowed_locals)
        return str(result)
    except Exception as e:
        return f"Error evaluating expression '{expression}': {e}"

model = init_chat_model(
    model=os.environ["FLEXAI_MODEL"],
    base_url=os.environ["FLEXAI_URL"],
    api_key=SecretStr(os.environ["FLEXAI_API_KEY"])
)

## Subagents as tools

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

research_agent = create_agent(model=model, tools=[web_search])
math_agent = create_agent(model=model, tools=[calculator])

@tool
def research(question: str) -> str:
    """Delegate research to the research specialist."""
    out = research_agent.invoke({"messages": [{"role": "user", "content": question}]})
    return out["messages"][-1].content

supervisor = create_agent(model=model, tools=[research])

## Handoff via a Command

In [ ]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command

@tool
def transfer_to_billing(runtime: ToolRuntime) -> Command:
    """Hand the conversation to the billing agent."""
    return Command(update={"active_agent": "billing"})

## Supervisor with two specialists
The supervisor sees only each subagent's final answer — their intermediate steps stay isolated.

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

research_agent = create_agent(model=model, tools=[web_search])
math_agent = create_agent(model=model, tools=[calculator])

def ask(agent, text): return agent.invoke(
    {"messages": [{"role": "user", "content": text}]})["messages"][-1].content

@tool
def research(question: str) -> str:
    """Delegate research to the research specialist."""
    return ask(research_agent, question)

@tool
def calculate(problem: str) -> str:
    """Delegate math to the math specialist."""
    return ask(math_agent, problem)

supervisor = create_agent(
    model=model,
    tools=[research, calculate],
    system_prompt="Route each part of the task to the right specialist, then combine.",
)
print(ask(supervisor, "Find last year's revenue and compute 12% growth."))

## Router: classify, then dispatch
A cheap classifier picks the specialist up front — one hop instead of a full tool loop.

In [ ]:
from langchain.chat_models import init_chat_model

classifier = init_chat_model("claude-haiku-4")
specialists = {"billing": billing_agent, "tech": tech_agent, "sales": sales_agent}

def route(question: str) -> str:
    label = classifier.invoke(
        f"Reply with one word — billing, tech, or sales.\n\n{question}"
    ).text.strip().lower()
    agent = specialists.get(label, tech_agent)
    return ask(agent, question)

print(route("My invoice looks wrong this month."))   # -> billing

## Handoff with shared state
A tool transfers control by writing to state; the next turn routes to the new agent.

In [ ]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command

@tool
def transfer_to_billing(reason: str, runtime: ToolRuntime) -> Command:
    """Hand off to the billing agent with a reason."""
    return Command(update={"active_agent": "billing", "handoff_reason": reason})

triage = create_agent(model="claude-sonnet-4-6", tools=[transfer_to_billing])

## Run specialists in parallel
When domains are independent, fan out and gather — much faster than a serial loop.

In [ ]:
import asyncio

async def gather_all(question: str) -> dict:
    research, numbers = await asyncio.gather(
        research_agent.ainvoke({"messages": [{"role": "user", "content": question}]}),
        math_agent.ainvoke({"messages": [{"role": "user", "content": question}]}),
    )
    return {
        "research": research["messages"][-1].content,
        "numbers": numbers["messages"][-1].content,
    }

print(asyncio.run(gather_all("Summarize the market and size it.")))